## TP4

# Hyper-paramétrage des méthodes d'apprentissage

Intéressons nous au paramétrage automatique des méthodes de classification ie. à la valeur de leurs hyper-paramètres. Ici, nous illustrerons ces techniques sur les _random forest_ vues lors du TP précédent.

Les hyper-paramètres des méthodes de classification ont un impact important sur leurs performances. En effet, comme vu dans le TP précédent, le nombre d'arbres d'une random forest ou bien l'utilisation d'un sous-ensemble d'observations ou de features impactent fortement les résultats obtenus. De plus, les hyper-paramètres peuvent être de nature différente.

Dans ce TP, nous nous concentrerons sur les méthodes GridSearch et RandomSearch qui sont les plus connues. 



## Préliminaires

Tout d'abord, il est important de bien comprendre les deux méthodes de paramétrage GridSearch et RandomSearch. Le module Scikit learn propose une documentation.

**§ En utilisant la documentation de Scikit Learn et vos ressources propres, décrire le principe des méthodes GridSearch et RandomSearch**


### GridSearch

La méthode **GridSearch** consiste à définir à l'avance un ensemble de valeurs possibles pour chaque hyper-paramètre du modèle. Toutes les combinaisons possibles de ces valeurs sont ensuite testées.

Pour chaque combinaison, le modèle est généralement évalué à l'aide d'une **validation croisée**. La combinaison obtenant les meilleures performances selon la métrique choisie, par exemple l'accuracy, est alors sélectionnée.

Cette méthode permet donc une recherche exhaustive, mais elle peut devenir très coûteuse lorsque le nombre d'hyper-paramètres ou de valeurs à tester augmente.

### RandomSearch

La méthode **RandomSearch** repose sur le même principe d'évaluation, mais au lieu de tester toutes les combinaisons possibles, elle sélectionne **aléatoirement un nombre défini de combinaisons d'hyper-paramètres**.

Elle permet ainsi d'explorer plus rapidement un espace de recherche important. Elle peut notamment tester davantage de valeurs différentes pour chaque hyper-paramètre avec un coût de calcul limité.

En résumé, **GridSearch effectue une recherche exhaustive sur une grille prédéfinie**, tandis que **RandomSearch explore aléatoirement une partie de l'espace des hyper-paramètres**.

**§ Expliquer les avantages/inconvénients de ces méthodes l'une par rapport à l'autre** 

**GridSearch** a pour principal avantage d'être **exhaustif** : toutes les combinaisons d'hyper-paramètres définies dans la grille sont testées. Il permet donc d'identifier la meilleure combinaison parmi celles proposées. En revanche, son principal inconvénient est son **coût de calcul**, qui peut devenir très important lorsque le nombre d'hyper-paramètres ou de valeurs testées augmente.

**RandomSearch** est généralement **plus rapide et moins coûteux**, car il ne teste qu'un nombre limité de combinaisons choisies aléatoirement. Il est particulièrement intéressant lorsque l'espace des hyper-paramètres est grand. En revanche, comme toutes les combinaisons ne sont pas testées, il existe un risque de **ne pas tester la meilleure configuration possible**.

Ainsi, **GridSearch est préférable lorsque l'espace de recherche est petit**, tandis que **RandomSearch est souvent plus adapté lorsque le nombre de paramètres et de valeurs possibles est important**.

Dans ce TP, nous focaliserons l'étude sur le paramétrage des _Random Forest_ du module de Sklearn vu au TP précédent.

**§ Identifier les types existants des paramètres des random forest de Sklearn**

Les paramètres de `RandomForestClassifier` peuvent être classés selon plusieurs types :

* **Paramètres entiers / discrets** : ils prennent un nombre entier de valeurs.

  * `n_estimators` : nombre d'arbres de la forêt ;
  * `max_depth` : profondeur maximale des arbres ;
  * `max_leaf_nodes` : nombre maximal de feuilles ;
  * `random_state` : graine utilisée pour contrôler l'aléatoire.

* **Paramètres réels / continus** :

  * `min_weight_fraction_leaf` ;
  * `min_impurity_decrease` ;
  * `ccp_alpha`.

* **Paramètres catégoriels** : choix parmi un ensemble de valeurs prédéfinies.

  * `criterion` : `"gini"`, `"entropy"` ou `"log_loss"` ;
  * `max_features` peut notamment prendre `"sqrt"` ou `"log2"`;
  * `class_weight` peut notamment prendre `"balanced"` ou `"balanced_subsample"`.

* **Paramètres booléens** :

  * `bootstrap` ;
  * `oob_score` ;
  * `warm_start`.

Certains paramètres peuvent avoir **plusieurs types**. Par exemple, `min_samples_split` peut être un entier ou un réel, et `max_features` peut être un entier, un réel, une chaîne de caractères ou `None`.

Cette diversité est importante pour l'hyper-paramétrage : selon le type du paramètre, il faudra définir soit une liste de valeurs possibles, soit un intervalle ou une distribution de valeurs à explorer.

**§ Indiquer l'impact de ces types sur l'utilisation des méthodes GridSearch et RandomSearch**

Le type des hyper-paramètres influence directement la manière dont ils peuvent être explorés par **GridSearch** et **RandomSearch**.

Avec **GridSearch**, il faut fournir une liste finie de valeurs à tester pour chaque paramètre. Cette méthode est donc bien adaptée aux paramètres **booléens** ou **catégoriels**, qui possèdent naturellement un nombre limité de valeurs possibles. Pour les paramètres entiers ou continus, il faut choisir manuellement quelques valeurs représentatives, ce qui limite la finesse de la recherche.

Avec **RandomSearch**, il est possible de fournir soit une liste de valeurs, soit une **distribution** dans laquelle les valeurs seront tirées aléatoirement. Cette méthode est donc particulièrement adaptée aux paramètres **numériques**, notamment continus ou pouvant prendre beaucoup de valeurs, car elle permet d'explorer un espace beaucoup plus large sans tester toutes les possibilités.

Ainsi, les paramètres catégoriels ou booléens sont facilement traités par les deux méthodes, tandis que les paramètres numériques avec un grand nombre de valeurs possibles sont généralement mieux adaptés à RandomSearch.

Dans la suite du TP, nous condidérerons que les paramètres suivants :

```
    n_estimators
    max_depth
    max_features
    bootstrap
```

**§ Décrire ces paramètres et leur ensemble de valeurs**

Les quatre hyper-paramètres étudiés sont les suivants :

* **`n_estimators`** : correspond au **nombre d'arbres** présents dans la forêt.
  Il prend des valeurs entières strictement positives : `1, 2, 3, ...`. Sa valeur par défaut est `100`. Augmenter ce nombre permet généralement de rendre les prédictions plus stables, mais augmente également le temps de calcul.

* **`max_depth`** : représente la **profondeur maximale de chaque arbre**.
  Il peut prendre une valeur entière positive ou `None`. Lorsque `max_depth=None`, les arbres continuent de se développer jusqu'à ce que les feuilles soient suffisamment pures ou qu'un autre critère d'arrêt soit atteint. Une faible profondeur limite la complexité des arbres et peut réduire le sur-apprentissage.

* **`max_features`** : définit le **nombre de variables examinées lors de la recherche de la meilleure séparation à chaque nœud**.
  Il peut prendre plusieurs types de valeurs :

  * un entier : nombre exact de features ;
  * un flottant compris entre `0` et `1` : proportion des features ;
  * `"sqrt"` : environ √n features ;
  * `"log2"` : environ log₂(n) features ;
  * `None` : toutes les features sont utilisées.

  La valeur par défaut actuelle est `"sqrt"`.

* **`bootstrap`** : indique si chaque arbre doit être entraîné sur un **échantillon bootstrap**, c'est-à-dire un échantillon obtenu par tirage avec remise dans le jeu d'entraînement.
  Il s'agit d'un booléen pouvant prendre deux valeurs :

  * `True` : utilisation d'échantillons bootstrap ;
  * `False` : chaque arbre utilise l'ensemble du jeu d'entraînement.

  Sa valeur par défaut est `True`.

## Méthode Grid Search

Utilisons le module [sklearn.model_selection.GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) pour paramétrer un random forest avec la méthode GridSearch.

Ici, la méthode GridSearch implémente une validation croisée afin de tester la performance d'une configuration (ie. un jeu de paramètres).

**§ Expliquer en quoi la validation croisée est adaptée dans ce cas**


Cela permet de gagner du temps. On va faire seulement une CV à 5 plis plutôt que de faire varier 30 fois le random state. 

**§ Proposer un protocole qui permet de trouver une bonne configuration d'un random forest en utilisant le module GridSearchCV**

## Etape 1 : Création des variables à étudier

```py
n_estimators = [30, 100, 500, 1000]
max_depth = [3, 7, 10]
max_features = ["sqrt", "log2", 0.2, 0.4, 0.6, 0.8, 1]
bootstrap = [True, False]
```

## Etape 2 : Cross-validation

&rarr; on choisi une CV à 5 plis. Ici nous avons rien à changer c'est la valeur par défault. 

## Etape 3 : RandomForestClassifier

Pour chaqu'une des configurations possible, lancer l'implémentation du modèle et sauvegarde les résultats. 

## Etape 4 : Test sur les résultats

On va utiliser le test de Friedman et Wilconxon pour voir quelles sont les meilleurs modèles.

En particulier, n'oubliez pas de bien définir l'espace des configurations.

**§ Appliquer ce protocole sur le jeu de données digits et analyser les résultats**

In [ ]:
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

PARAM_GRID = {
    "n_estimators" : [30, 100, 500, 1000],
    "max_depth" : [3, 7, 10],
    "max_features" : ["sqrt", "log2", 0.2, 0.4, 0.6, 0.8, 1.0],
    "bootstrap" : [True, False]
}

j_digits = load_digits()
X, Y = j_digits.data, j_digits.target

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=RANDOM_STATE), param_grid=PARAM_GRID, n_jobs=-1)
grid_search.fit(X, Y)

In [ ]:
print("Meilleurs paramètres :", grid_search.best_params_)
print("Accuracy CV :", grid_search.best_score_)

Meilleurs paramètres : {'bootstrap': False, 'max_depth': 10, 'max_features': 'log2', 'n_estimators': 500}
Accuracy CV : 0.9460368307025689
                                                params  ...  rank_test_score
146  {'bootstrap': False, 'max_depth': 10, 'max_fea...  ...                1
147  {'bootstrap': False, 'max_depth': 10, 'max_fea...  ...                2
65   {'bootstrap': True, 'max_depth': 10, 'max_feat...  ...                3
149  {'bootstrap': False, 'max_depth': 10, 'max_fea...  ...                4
151  {'bootstrap': False, 'max_depth': 10, 'max_fea...  ...                5
63   {'bootstrap': True, 'max_depth': 10, 'max_feat...  ...                5
59   {'bootstrap': True, 'max_depth': 10, 'max_feat...  ...                7
58   {'bootstrap': True, 'max_depth': 10, 'max_feat...  ...                8
150  {'bootstrap': False, 'max_depth': 10, 'max_fea...  ...                9
67   {'bootstrap': True, 'max_depth': 10, 'max_feat...  ...               10

[10 rows x 4 

## Méthode Random Search

Utilisons le module [sklearn.model_selection.RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) pour paramétrer un random forest avec la méthode RandomSearch.

**§ Proposer un protocole qui permet de trouver une bonne configuration d'un random forest en utilisant le module RandomSearchCV**

## Étape 1 : Création des distributions des variables à étudier

Pour `n_estimators`, on utilise une loi normale centrée autour de **500 arbres**.
Pour `max_depth`, on utilise une loi normale centrée autour d'une profondeur de **10**.

Comme ces deux paramètres doivent prendre des valeurs entières positives, les valeurs générées par les lois normales devront être converties en entiers et limitées afin d'éviter les valeurs invalides.

Par exemple :

```python
import numpy as np

rng = np.random.default_rng(42)

n_estimators = np.clip(
    rng.normal(loc=500, scale=200, size=1000).astype(int),
    10,
    1500
)

max_depth = np.clip(
    rng.normal(loc=10, scale=4, size=1000).astype(int),
    1,
    30
)

PARAM_RANDOM = {
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "max_features": ["sqrt", "log2", 0.2, 0.4, 0.6, 0.8, 1.0],
    "bootstrap": [True, False]
}
```

Ici :

* `n_estimators` est centré autour de `500` avec un écart-type de `200` ;
* `max_depth` est centré autour de `10` avec un écart-type de `4` ;
* `max_features` reste un ensemble de valeurs prédéfinies ;
* `bootstrap` reste un paramètre booléen.

## Étape 2 : Définition du nombre de configurations

Avec RandomSearch, il faut définir le nombre de configurations différentes que l'on souhaite tester grâce au paramètre `n_iter`.

Par exemple :

```python
n_iter = 100
```

Cela signifie que seulement **100 configurations aléatoires** seront évaluées, contrairement à GridSearch qui teste toutes les combinaisons possibles.

## Étape 3 : Cross-validation

Pour chaque configuration tirée aléatoirement, on réalise une **validation croisée à 5 plis**.

```python
cv=5
```

Les performances d'une configuration correspondent donc à la moyenne des accuracies obtenues sur les 5 plis.

## Étape 4 : RandomForestClassifier et RandomizedSearchCV

On utilise `RandomizedSearchCV` afin de tirer les configurations et d'évaluer chaque Random Forest :

```python
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=PARAM_RANDOM,
    n_iter=100,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, Y_train)
```

À la fin de la recherche, on peut récupérer la configuration ayant obtenu la meilleure accuracy moyenne :

```python
print(random_search.best_params_)
print(random_search.best_score_)
```

## Étape 5 : Analyse des résultats

Les configurations sont dans un premier temps classées selon leur **accuracy moyenne obtenue par validation croisée**.

On peut ensuite comparer les meilleures configurations à l'aide de tests statistiques.

Le **test de Friedman** permet de déterminer s'il existe une différence globale entre plusieurs configurations.

Si une différence significative est détectée, des **tests de Wilcoxon** peuvent ensuite être réalisés deux à deux afin d'identifier les configurations dont les performances diffèrent significativement.

Enfin, la configuration retenue est évaluée sur le jeu de test, qui n'a pas été utilisé pendant la recherche des hyper-paramètres.


**§ Appliquer ce protocole sur le jeu de données digits**

In [30]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

rng = np.random.default_rng(RANDOM_STATE)

n_estimators = np.clip(
    rng.normal(loc=500, scale=200, size=1000).astype(int),
    10,
    1500
)

max_depth = np.clip(
    rng.normal(loc=10, scale=4, size=1000).astype(int),
    1,
    30
)

PARAM_RANDOM = {
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "max_features": ["sqrt", "log2", 0.2, 0.4, 0.6, 0.8, 1.0],
    "bootstrap": [True, False]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=PARAM_RANDOM,
    n_iter=100,
    scoring="accuracy",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_search.fit(X, Y)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=2),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': array([ 8,  1,  8, 14, 10,  6, 12, 12, 18,  7,  8, 10,  9, 10,  9,  7,  3,
        7,  4,  5,  7, 12, 13,  5, 11, 13,  9,  3,  1, 13, 13, 11,  5, 15,
       14,  9,  9,  2, 12,  3, 12, 12, 10, 16,  8, 12, 13, 10,  8, 14, 11,
        9,  9, 10, 19, 10,  8,  5,  5,  2, 10, 10,  5,  3,  8,  7,  4, 11,
       10,  9, 12,  7, 15,  5, 11, 12,  8,...
        548,  245,  775,  341,  487,  441,  318,  766,  619,  377,  513,
        413,  623,  424,  346,  371,  412,  557,  316,  482,  588,  967,
        671,   10,  397,  420,  564,  813,  648,   39,  561,  521,  582,
        771,  281,  720,  169,  510,  576,  956,  359,  161,  637,  367,
        812,  358,  324,  748,  517,  700,  213,  830,  382,  663,  299,
        189,  448,  534,  815,  570,  643,  631, 1122,  336,  751,  470,
        552,  390,  732,  249,  235,  458,  268,  640,  684,  349,  656,
        367,  491,  501,  875,  499,  317,  316,  739,  166,  405])},
                   random_state=2, scoring='accuracy')

In [23]:
print(random_search.best_params_)
print(random_search.best_score_)

{'n_estimators': np.int64(518), 'max_features': 'log2', 'max_depth': np.int64(12), 'bootstrap': False}
0.9471541318477252


**§ Comparer les résultats obtenus avec les méthodes GridSearch et RandomSearch puis discuter des résultats**

## Hybridation Grid Search et Random Search

Pour terminer, nous allons utiliser les forces de chacune de ces méthodes pour pouvoir affiner le paramétrage de la _Random Forest_

**§ Proposer un nouvel espace de configuration, plus large**

In [29]:
from scipy.stats import randint

PARAM_RANDOM = {
    "n_estimators": randint(50, 2001),
    "max_depth": list(range(2, 31)) + [None],
    "max_features": [
        "sqrt", "log2",
        0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9, 1.0
    ],
    "bootstrap": [True, False]
}


**§ Proposer un protocole expérimental utilisant Grid Search et Random Search**

Dans un premier temps, un Random Search est réalisé sur un espace d'hyper-paramètres relativement large. Cette méthode permet d'explorer efficacement un grand nombre de valeurs sans tester toutes les combinaisons possibles.

Une fois une zone de bonnes performances identifiée, un Grid Search est effectué autour de la meilleure configuration obtenue. Cette deuxième étape permet d'explorer de manière exhaustive et plus fine le voisinage de la solution trouvée.

Pour chaque configuration, une validation croisée à 5 plis est utilisée et l'accuracy moyenne sert de critère de comparaison. Le jeu de test est conservé à part et n'est utilisé qu'après la sélection finale des hyper-paramètres.


**§ Appliquer ce protocole sur le jeu de données digits et anlayser les résultats**

In [24]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=PARAM_RANDOM,
    n_iter=100,
    cv=5,
    scoring="accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_search.fit(X, Y)

print(random_search.best_params_)
print(random_search.best_score_)

{'n_estimators': np.int64(562), 'max_features': 'log2', 'max_depth': np.int64(10), 'bootstrap': False}
0.9454812751470133


In [ ]:
PARAM_GRID_FINE = {
    "n_estimators": [i for i in range(510, 560, 2)],
    "max_depth": [9, 10, 11, 12],
    "max_features": ["log2"],
    "bootstrap": [False]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=PARAM_GRID_FINE,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X, Y)

print("Meilleurs paramètres :", grid_search.best_params_)
print("Accuracy CV :", grid_search.best_score_)

Meilleurs paramètres : {'bootstrap': False, 'max_depth': 10, 'max_features': 'log2', 'n_estimators': 538}
Accuracy CV : 0.9465923862581244
